# Fine-Tuning Indonesian Emotion Classification Model on Kaggle GPU

Notebook ini dibuat otomatis untuk melatih model klasifikasi emosi (7 kelas: Marah, Senang, Sedih, Takut, Jijik, Terkejut, Netral) bahasa Indonesia menggunakan arsitektur IndoBERT. Notebook ini dirancang agar siap dijalankan di Kaggle menggunakan akselerasi GPU (T4 x2 atau P100).

### Alur Langkah:
1. Install/Update dependencies otomatis
2. Load dataset dari file input Kaggle
3. Preprocessing data & standardisasi label
4. Fine-tuning menggunakan HuggingFace Trainer API dengan optimisasi FP16
5. Evaluasi model (Accuracy, Precision, Recall, F1, Confusion Matrix)
6. Ekspor model ZIP ke direktori output Kaggle `/kaggle/working/`

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn openpyxl matplotlib seaborn

## 2. Load Dataset

Unggah dataset Anda ke Kaggle (Add Data -> Upload a dataset), lalu sesuaikan path ke file dataset CSV/XLSX Anda di bawah.

In [ ]:
import pandas as pd
import numpy as np
import os

# Jalankan ini untuk list input files jika Anda lupa path-nya
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Ganti dengan path dataset Anda di Kaggle
dataset_path = '/kaggle/input/emotion-dataset.csv' 

if not os.path.exists(dataset_path):
    print(f"Dataset tidak ditemukan di: {dataset_path}. Harap sesuaikan path atau gunakan dataset dummy.")
    # Dataset dummy jika file tidak ditemukan
    df = pd.DataFrame({
        'text': ['saya marah sekali dengannya', 'hari ini menyenangkan sekali', 'biasa saja'],
        'label': ['marah', 'senang', 'netral']
    })
else:
    if dataset_path.endswith('.csv'):
        df = pd.read_csv(dataset_path)
    else:
        df = pd.read_excel(dataset_path)

print(f"\nDataset berhasil dimuat: {len(df)} baris.")
print(df.head())

## 3. Preprocessing Data & Label Mapping

In [ ]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove URLs
    text = re.sub(r'https?:\/\/\S+|www\.\S+', '', text)
    # Remove mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags symbol
    text = text.replace('#', '')
    # Remove emojis & non-ascii
    text = text.encode('ascii', 'ignore').decode('ascii')
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

# Sesuaikan nama kolom jika berbeda
text_col = 'text'
label_col = 'label'

df['clean_text'] = df[text_col].apply(clean_text)

# Mapping label ke integer
label_map = {
    'marah': 0, 'Marah': 0, 'angry': 0,
    'senang': 1, 'Senang': 1, 'happy': 1, 'bahagia': 1,
    'sedih': 2, 'Sedih': 2, 'sad': 2,
    'takut': 3, 'Takut': 3, 'fear': 3,
    'jijik': 4, 'Jijik': 4, 'disgust': 4,
    'terkejut': 5, 'Terkejut': 5, 'surprise': 5, 'kaget': 5,
    'netral': 6, 'Netral': 6, 'neutral': 6
}
df['label_id'] = df[label_col].map(label_map)
df = df.dropna(subset=['label_id'])
df['label_id'] = df['label_id'].astype(int)

print("Distribusi Label:")
print(df['label_id'].value_counts())

## 4. Train / Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42,
    stratify=df['label_id']
)

print(f"Jumlah data training: {len(train_df)}")
print(f"Jumlah data validation: {len(val_df)}")

## 5. Tokenisasi Dataset

In [ ]:
from transformers import AutoTokenizer
import torch
from datasets import Dataset

model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(batch['clean_text'], truncation=True, padding='max_length', max_length=128)

train_dataset = Dataset.from_pandas(train_df[['clean_text', 'label_id']].rename(columns={'label_id': 'label'}))
val_dataset = Dataset.from_pandas(val_df[['clean_text', 'label_id']].rename(columns={'label_id': 'label'}))

train_tokenized = train_dataset.map(tokenize_fn, batched=True)
val_tokenized = val_dataset.map(tokenize_fn, batched=True)

## 6. Fine-Tuning Model

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_tuple

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_tuple(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Load Model dengan 7 Label kelas
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=7)

training_args = TrainingArguments(
    output_dir='/kaggle/working/results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir='/kaggle/working/logs',
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# Mulai Training
trainer.train()

## 7. Evaluasi & Visualisasi

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

emotion_names = ["Marah", "Senang", "Sedih", "Takut", "Jijik", "Terkejut", "Netral"]

# Ambil prediksi
preds_output = trainer.predict(val_tokenized)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

# Print metrik
print(classification_report(y_true, y_pred, target_names=emotion_names))

# Render Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=emotion_names,
            yticklabels=emotion_names)
plt.xlabel('Prediksi')
plt.ylabel('Sebenarnya')
plt.title('Confusion Matrix - Emotion')
plt.show()

## 8. Simpan & Ekspor Model

In [ ]:
import json

# Simpan model lokal ke working directory Kaggle
model_save_path = '/kaggle/working/emotion_model'
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

# Simpan label mapping
label_mapping = {
    "label2id": {"Marah": 0, "Senang": 1, "Sedih": 2, "Takut": 3, "Jijik": 4, "Terkejut": 5, "Netral": 6},
    "id2label": {"0": "Marah", "1": "Senang", "2": "Sedih", "3": "Takut", "4": "Jijik", "5": "Terkejut", "6": "Netral"},
    "labels": ["Marah", "Senang", "Sedih", "Takut", "Jijik", "Terkejut", "Netral"]
}
with open(os.path.join(model_save_path, 'label_mapping.json'), 'w') as f:
    json.dump(label_mapping, f, indent=2)

# Zip model ke output directory
!zip -r /kaggle/working/emotion_model.zip /kaggle/working/emotion_model
print("\nModel sukses di-zip ke: /kaggle/working/emotion_model.zip. Anda dapat mengunduhnya langsung dari tab Output Kaggle!")